In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ========== 数据准备 ==========
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [2]:
class CIFAR10CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [3]:
# total = sum(p.numel() for p in model.parameters())
# print(f"总参数量: {total:,}")

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
device

device(type='cuda')

In [6]:
print(f"使用设备: {device}")

使用设备: cuda


In [7]:
model = CIFAR10CNN().to(device)

In [8]:
total = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total:,}")

总参数量: 667,178


In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [10]:
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

In [13]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss, correct, total = 0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

In [14]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)

        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return correct / total

In [15]:
num_epochs = 30
for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_acc = evaluate(model, test_loader, device)

    scheduler.step()

    print(f"Epoch {epoch:2d}/{num_epochs} | "
          f"Loss: {train_loss:.4f} | "
          f"Train Acc: {train_acc:.2%} | "
          f"Test Acc: {test_acc:.2%}")

Epoch  1/30 | Loss: 1.4982 | Train Acc: 44.97% | Test Acc: 60.40%
Epoch  2/30 | Loss: 1.1326 | Train Acc: 59.62% | Test Acc: 58.81%
Epoch  3/30 | Loss: 0.9878 | Train Acc: 65.08% | Test Acc: 68.74%
Epoch  4/30 | Loss: 0.8967 | Train Acc: 68.43% | Test Acc: 63.75%
Epoch  5/30 | Loss: 0.8447 | Train Acc: 70.58% | Test Acc: 76.23%
Epoch  6/30 | Loss: 0.7843 | Train Acc: 72.70% | Test Acc: 76.04%
Epoch  7/30 | Loss: 0.7375 | Train Acc: 74.59% | Test Acc: 72.35%
Epoch  8/30 | Loss: 0.7140 | Train Acc: 75.42% | Test Acc: 78.27%
Epoch  9/30 | Loss: 0.6795 | Train Acc: 76.88% | Test Acc: 78.84%
Epoch 10/30 | Loss: 0.6505 | Train Acc: 77.76% | Test Acc: 79.61%
Epoch 11/30 | Loss: 0.6188 | Train Acc: 78.84% | Test Acc: 80.91%
Epoch 12/30 | Loss: 0.5982 | Train Acc: 79.63% | Test Acc: 80.19%
Epoch 13/30 | Loss: 0.5812 | Train Acc: 80.42% | Test Acc: 82.23%
Epoch 14/30 | Loss: 0.5569 | Train Acc: 81.22% | Test Acc: 80.51%
Epoch 15/30 | Loss: 0.5391 | Train Acc: 81.82% | Test Acc: 82.65%
Epoch 16/3

In [16]:
torch.save(model, "cifar10_cnn_full_model.pth")

In [17]:
torch.save(model.state_dict(), "cifar10_cnn_weights.pth")
print("模型权重保存成功")

模型权重保存成功
